# Scaffold Nonchiral Preparation

Recovered workflow. See `docs/WORKFLOW_ORDER.md` for dependencies and execution restrictions. Source cell numbers below are zero-based.


Source cell: 51.


In [ ]:
# ============================================================
# Reconstruct corrected NON-CHIRAL scaffold protocol
# Original files remain unchanged. No training.
# ============================================================

from pathlib import Path
from datetime import datetime
import hashlib
import json
import shutil
import importlib.metadata

import numpy as np
import pandas as pd
from IPython.display import display, FileLink

# ---------- 1. Environment ----------
try:
    from rdkit import Chem, rdBase
    from rdkit.Chem.Scaffolds import MurckoScaffold
except ImportError:
    raise RuntimeError(
        "RDKit is missing. Run this in a separate cell:\n"
        "%pip install rdkit==2026.3.6\n"
        "Then rerun this cell."
    )

print("RDKit:", rdBase.rdkitVersion, flush=True)
assert rdBase.rdkitVersion == "2026.03.6", (
    "Historical corrected experiment used RDKit 2026.03.6. "
    "Run %pip install rdkit==2026.3.6 in a separate cell, "
    "restart the kernel if RDKit was already imported, then rerun."
)

# ---------- 2. Input ----------
SOURCE = Path(
    "/kaggle/input/datasets/amiramohammeddd/"
    "ddi-unique263-cal-test-stage2/"
    "DDI_V2_SCAFFOLD_DISJOINT_84_READY_FOR_TRAINING_20260906_2224"
)

FILES = {
    "train": "scaffold_disjoint_train.csv",
    "calibration": "scaffold_disjoint_calibration.csv",
    "test": "scaffold_disjoint_strict_test.csv",
}

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

splits = {}
source_hashes = {}

for split, filename in FILES.items():
    path = SOURCE / filename
    assert path.is_file(), f"Missing: {path}"

    source_hashes[filename] = sha256_file(path)
    df = pd.read_csv(path, keep_default_na=False)

    required = {"Drug1", "Drug2", "Label", "label_id"}
    assert required.issubset(df.columns), (split, df.columns.tolist())
    assert df["Drug1"].map(lambda x: isinstance(x, str) and bool(x)).all()
    assert df["Drug2"].map(lambda x: isinstance(x, str) and bool(x)).all()

    splits[split] = df

expected_original_counts = {
    "train": 91477,
    "calibration": 16151,
    "test": 12242,
}

for name, expected in expected_original_counts.items():
    assert len(splits[name]) == expected, (
        name, len(splits[name]), expected
    )

print("\nOriginal split counts verified.", flush=True)

# ---------- 3. Molecular identities and scaffolds ----------
all_drugs = sorted({
    drug
    for df in splits.values()
    for col in ("Drug1", "Drug2")
    for drug in df[col]
})

scaffolds = {}
canonical = {}
invalid = []

print(f"Checking {len(all_drugs):,} molecules...", flush=True)

for index, smiles in enumerate(all_drugs, start=1):
    mol = Chem.MolFromSmiles(smiles)

    if mol is None or mol.GetNumAtoms() == 0:
        invalid.append(smiles)
        continue

    canonical[smiles] = Chem.MolToSmiles(
        mol, canonical=True, isomericSmiles=True
    )

    # Empty scaffold is retained as a shared key for acyclic molecules.
    # No per-drug fallback or silent exclusion.
    scaffolds[smiles] = MurckoScaffold.MurckoScaffoldSmiles(
        mol=mol,
        includeChirality=False,
    )

    if index % 250 == 0 or index == len(all_drugs):
        print(f"Checked {index:,}/{len(all_drugs):,}", flush=True)

assert not invalid, (
    f"{len(invalid)} invalid/empty molecules. "
    f"Stop for review. Examples: {invalid[:5]}"
)

def drug_set(df):
    return set(df["Drug1"]) | set(df["Drug2"])

def scaffold_set(df):
    return {scaffolds[drug] for drug in drug_set(df)}

development_scaffolds = (
    scaffold_set(splits["train"])
    | scaffold_set(splits["calibration"])
)

# ---------- 4. Correct Test only ----------
original_test = splits["test"]

s1 = original_test["Drug1"].map(scaffolds)
s2 = original_test["Drug2"].map(scaffolds)

overlap1 = s1.isin(development_scaffolds)
overlap2 = s2.isin(development_scaffolds)
keep = ~(overlap1 | overlap2)

corrected_test = original_test.loc[keep].copy().reset_index(drop=True)

observed_before = set(original_test["label_id"])
observed_after = set(corrected_test["label_id"])

print("\nRECONSTRUCTED TEST")
print("Original rows:", len(original_test))
print("Retained rows:", len(corrected_test))
print("Excluded rows:", int((~keep).sum()))
print("Observed classes:", len(observed_after))
print("Lost encoded label IDs:", sorted(observed_before - observed_after))

# Historical results are a reproduction gate, not a target to force.
assert len(corrected_test) == 11142, (
    "Retained count differs from historical 11,142. "
    "No protocol saved; inspect scaffold definition and source files."
)
assert len(observed_after) == 56, (
    "Observed-class count differs from historical 56."
)

corrected = {
    "train": splits["train"].copy(),
    "calibration": splits["calibration"].copy(),
    "test": corrected_test,
}

# Explicit names distinguish recomputed keys from old stored scaffold columns.
for df in corrected.values():
    df["scaffold_nonchiral_1"] = df["Drug1"].map(scaffolds)
    df["scaffold_nonchiral_2"] = df["Drug2"].map(scaffolds)

# Confirm every original column and row order is preserved for development.
for name in ("train", "calibration"):
    pd.testing.assert_frame_equal(
        corrected[name][splits[name].columns],
        splits[name],
    )

# ---------- 5. Isolation audit ----------
def canonical_pair_keys(df):
    return {
        tuple(sorted((canonical[a], canonical[b])))
        for a, b in zip(df["Drug1"], df["Drug2"])
    }

audit = {}
pairs = {name: canonical_pair_keys(df) for name, df in corrected.items()}

for left, right in (
    ("train", "calibration"),
    ("train", "test"),
    ("calibration", "test"),
):
    key = f"{left}_{right}_canonical_pair_overlap"
    audit[key] = len(pairs[left] & pairs[right])
    assert audit[key] == 0, (key, audit[key])

for development in ("train", "calibration"):
    key = f"{development}_test_nonchiral_scaffold_overlap"
    audit[key] = len(
        scaffold_set(corrected[development])
        & scaffold_set(corrected["test"])
    )
    assert audit[key] == 0, (key, audit[key])

print("\nIsolation checks passed:", audit, flush=True)

# ---------- 6. Save a NEW protocol ----------
stamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
OUT = Path("/kaggle/working") / (
    f"DDI_SCAFFOLD_NONCHIRAL_RECONSTRUCTED_{stamp}"
)
OUT.mkdir(parents=True, exist_ok=False)

saved_files = {
    "train": "train.csv",
    "calibration": "calibration.csv",
    "test": "strict_test.csv",
}

for name, df in corrected.items():
    path = OUT / saved_files[name]
    df.to_csv(path, index=False)

    restored = pd.read_csv(path, keep_default_na=False)
    pd.testing.assert_frame_equal(
        df.reset_index(drop=True),
        restored,
        check_dtype=False,
    )

# Save exact source-row membership for every split.
manifest_rows = []
for name in ("train", "calibration"):
    manifest_rows.extend({
        "split": name,
        "saved_row": i,
        "source_row": i,
        "source_file": FILES[name],
    } for i in range(len(corrected[name])))

manifest_rows.extend({
    "split": "test",
    "saved_row": i,
    "source_row": int(source_row),
    "source_file": FILES["test"],
} for i, source_row in enumerate(np.flatnonzero(keep.to_numpy())))

pd.DataFrame(manifest_rows).to_csv(
    OUT / "split_row_manifest.csv", index=False
)

excluded = original_test.loc[~keep].copy()
excluded.insert(0, "source_row", np.flatnonzero((~keep).to_numpy()))
excluded["drug1_scaffold_seen_in_development"] = overlap1.loc[~keep].to_numpy()
excluded["drug2_scaffold_seen_in_development"] = overlap2.loc[~keep].to_numpy()
excluded.to_csv(OUT / "excluded_test_rows.csv", index=False)

memberships = {name: drug_set(df) for name, df in corrected.items()}
pd.DataFrame([{
    "SMILES": drug,
    "canonical_smiles": canonical[drug],
    "scaffold_nonchiral": scaffolds[drug],
    **{f"in_{name}": drug in members
       for name, members in memberships.items()},
} for drug in all_drugs]).to_csv(
    OUT / "molecular_scaffold_audit.csv", index=False
)

support = pd.DataFrame({
    name: df["label_id"].value_counts()
    for name, df in corrected.items()
}).reindex(range(84), fill_value=0).fillna(0).astype(int)
support.index.name = "label_id"
support.to_csv(OUT / "class_support.csv")

summary = [{
    "split": name,
    "rows": len(df),
    "classes": df["label_id"].nunique(),
    "drugs": len(drug_set(df)),
} for name, df in corrected.items()]

metadata = {
    "protocol": "test_both_nonchiral_scaffolds_unseen_in_development",
    "rdkit_version": rdBase.rdkitVersion,
    "scaffold_definition": "Bemis-Murcko; includeChirality=False",
    "empty_scaffold_policy": "Retain empty string as shared scaffold key",
    "source_folder": str(SOURCE),
    "source_sha256": source_hashes,
    "splits": summary,
    "original_test_rows": len(original_test),
    "excluded_test_rows": len(excluded),
    "lost_encoded_label_ids": [
        int(x) for x in sorted(observed_before - observed_after)
    ],
    "isolation_audit": audit,
    "old_scaffold_columns": (
        "Preserved as historical data; use scaffold_nonchiral_1/2."
    ),
    "preprocessing_status": "not_built",
    "training_performed": False,
    "historical_reproduction": (
        "Counts reproduced; identity with missing historical corrected "
        "files cannot be verified without their hashes."
    ),
    "saved_csv_sha256": {
        p.name: sha256_file(p) for p in sorted(OUT.glob("*.csv"))
    },
}

(OUT / "protocol_metadata.json").write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

# ---------- 7. Backup ----------
zip_path = Path(shutil.make_archive(str(OUT), "zip", root_dir=OUT))

import zipfile
with zipfile.ZipFile(zip_path) as archive:
    bad = archive.testzip()
    assert bad is None, f"Corrupt ZIP entry: {bad}"

SCAFFOLD_PROTOCOL_DIR = OUT

print("\n CORRECTED SCAFFOLD PROTOCOL SAVED")
display(pd.DataFrame(summary))
print("Folder:", OUT)
print("ZIP:", zip_path)
print(" Train/Calibration original rows and values preserved.")
print(" Corrected Test: 11,142 rows; 56 observed classes.")
print(" No features or training yet.")

display(FileLink(str(zip_path.relative_to("/kaggle/working"))))

Source cell: 54.


In [ ]:
# ============================================================
# Scaffold molecular features — TRAIN-ONLY preprocessing
# Morgan 2048 + original 208 RDKit descriptor registry
# Fixed log1p(Ipc), train-median imputation, variance filtering,
# and StandardScaler fitted exclusively on training drugs.
# ============================================================

from pathlib import Path
from datetime import datetime
import hashlib
import json
import shutil
import zipfile
import warnings

import numpy as np
import pandas as pd
import sklearn
from sklearn.preprocessing import StandardScaler
from rdkit import Chem, rdBase
from rdkit.Chem import Descriptors, rdFingerprintGenerator
from tqdm.auto import tqdm
from IPython.display import display, FileLink

PROTOCOL = Path(
    "/kaggle/working/"
    "DDI_SCAFFOLD_NONCHIRAL_RECONSTRUCTED_20260921_190107_430125"
)
assert PROTOCOL.is_dir(), f"Missing: {PROTOCOL}"
assert rdBase.rdkitVersion == "2026.03.6", rdBase.rdkitVersion

def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

# Recover registry even after notebook variables are cleared.
registry_paths = sorted(
    Path("/kaggle/input").rglob("raw_molecular_features.npz")
)
registries = {}

for path in registry_paths:
    with np.load(path, allow_pickle=False) as saved:
        if "descriptor_names" not in saved.files:
            continue
        names = [
            x.decode("utf-8") if isinstance(x, bytes) else str(x)
            for x in saved["descriptor_names"].tolist()
        ]
    if len(names) == 208 and len(set(names)) == 208 and "Ipc" in names:
        registries.setdefault(tuple(names), []).append(path)

assert len(registries) == 1, (
    "Expected exactly one distinct historical 208-descriptor registry."
)
descriptor_names = list(next(iter(registries)))
registry_sources = registries[tuple(descriptor_names)]

functions = []
for name in descriptor_names:
    fn = getattr(Descriptors, name, None)
    assert callable(fn), f"Unavailable descriptor: {name}"
    functions.append(fn)

files = {
    "train": "train.csv",
    "calibration": "calibration.csv",
    "test": "strict_test.csv",
}

with (PROTOCOL / "protocol_metadata.json").open(encoding="utf-8") as f:
    protocol_meta = json.load(f)

splits = {}
for name, filename in files.items():
    path = PROTOCOL / filename
    assert sha256_file(path) == protocol_meta["saved_csv_sha256"][filename]
    splits[name] = pd.read_csv(path, keep_default_na=False)

members = {
    name: set(df["Drug1"]) | set(df["Drug2"])
    for name, df in splits.items()
}
all_drugs = sorted(set.union(*members.values()))
drug_to_row = {drug: i for i, drug in enumerate(all_drugs)}
train_rows = np.array(
    [drug_to_row[drug] for drug in sorted(members["train"])],
    dtype=np.int64,
)

assert len(all_drugs) == 1659, len(all_drugs)
assert len(train_rows) == 1276, len(train_rows)

OUT = PROTOCOL / (
    "train_only_ipc_log1p_"
    + datetime.now().strftime("%Y%m%d_%H%M%S_%f")
)
OUT.mkdir(parents=True, exist_ok=False)

print("RDKit:", rdBase.rdkitVersion)
print("Train fit drugs:", len(train_rows))
print("Cached drugs:", len(all_drugs))
print("Output:", OUT, flush=True)

# ---------- Compute molecule-local raw features ----------
generator = rdFingerprintGenerator.GetMorganGenerator(
    radius=2,
    fpSize=2048,
    includeChirality=False,
)
fingerprints = np.empty((len(all_drugs), 2048), dtype=np.uint8)
raw = np.full((len(all_drugs), 208), np.nan, dtype=np.float64)
errors = []

for i, smiles in enumerate(tqdm(all_drugs, desc="Molecular features")):
    mol = Chem.MolFromSmiles(smiles)
    assert mol is not None and mol.GetNumAtoms() > 0, (
        f"Invalid/empty molecule at row {i}: {smiles}"
    )
    fingerprints[i] = generator.GetFingerprintAsNumPy(mol)

    for j, fn in enumerate(functions):
        try:
            raw[i, j] = float(fn(mol))
        except Exception as exc:
            errors.append({
                "drug_row": i,
                "descriptor": descriptor_names[j],
                "error": f"{type(exc).__name__}: {exc}",
            })

np.savez_compressed(
    OUT / "raw_molecular_features.npz",
    all_drugs=np.asarray(all_drugs, dtype=str),
    descriptor_names=np.asarray(descriptor_names, dtype=str),
    train_drug_indices=train_rows,
    morgan=fingerprints,
    rdkit_raw=raw,
)

pd.DataFrame(
    errors, columns=["drug_row", "descriptor", "error"]
).to_csv(OUT / "descriptor_errors.csv", index=False)

# ---------- Fixed transformation; no fitted parameters ----------
ipc_col = descriptor_names.index("Ipc")

def transform_raw(values):
    transformed = np.asarray(values, dtype=np.float64).copy()
    finite = np.isfinite(transformed[:, ipc_col])
    assert (transformed[finite, ipc_col] >= 0).all(), "Negative Ipc."
    transformed[finite, ipc_col] = np.log1p(
        transformed[finite, ipc_col]
    )
    transformed[~np.isfinite(transformed)] = np.nan
    return transformed

transformed = transform_raw(raw)

# ---------- Fit preprocessing on Train only ----------
has_train_value = np.isfinite(transformed[train_rows]).any(axis=0)
available_indices = np.flatnonzero(has_train_value)
assert len(available_indices) > 0

train_available = transformed[train_rows][:, available_indices]
medians = np.nanmedian(train_available, axis=0)
assert np.isfinite(medians).all()

available = transformed[:, available_indices]
imputed = np.where(np.isfinite(available), available, medians)

with np.errstate(over="raise", invalid="raise"):
    train_variance = np.var(imputed[train_rows], axis=0)

assert np.isfinite(train_variance).all()
variable_mask = train_variance > 0.0
assert variable_mask.any()

selected_indices = available_indices[variable_mask]
selected = imputed[:, variable_mask]

scaler = StandardScaler()
scaler.fit(selected[train_rows])
scaled = scaler.transform(selected)

assert np.isfinite(scaled).all()
combined = np.concatenate(
    [fingerprints.astype(np.float64), scaled], axis=1
)
assert np.isfinite(combined).all()
assert np.abs(combined).max() <= np.finfo(np.float32).max
x_new = combined.astype(np.float32)

all_missing_dropped = [
    descriptor_names[i] for i in np.flatnonzero(~has_train_value)
]
zero_variance_dropped = [
    descriptor_names[i]
    for i in available_indices[~variable_mask]
]
retained_names = [descriptor_names[i] for i in selected_indices]

# Do not force the retained dimension to match the old experiment.
if len(retained_names) != 202:
    warnings.warn(
        f"Retained {len(retained_names)} descriptors; historical count was "
        "202. Inspect before training. No dimensions were forced."
    )

# ---------- Save portable preprocessing parameters ----------
np.savez_compressed(
    OUT / "preprocessing_parameters.npz",
    descriptor_names=np.asarray(descriptor_names, dtype=str),
    ipc_col=np.asarray(ipc_col),
    available_indices=available_indices,
    medians=medians,
    variable_mask=variable_mask,
    selected_indices=selected_indices,
    train_variance=train_variance,
    scaler_mean=scaler.mean_,
    scaler_scale=scaler.scale_,
    scaler_var=scaler.var_,
    train_drug_indices=train_rows,
)

np.savez_compressed(
    OUT / "molecular_features.npz",
    x=x_new,
    all_drugs=np.asarray(all_drugs, dtype=str),
    descriptor_names=np.asarray(retained_names, dtype=str),
    train_drug_indices=train_rows,
)

pd.DataFrame([{
    "drug_row": i,
    "SMILES": drug,
    **{f"in_{name}": drug in values for name, values in members.items()},
} for i, drug in enumerate(all_drugs)]).to_csv(
    OUT / "drug_membership.csv", index=False
)

# Preserve full source rows and append feature indices.
for name, df in splits.items():
    pairs = df.copy()
    pairs["drug1_id"] = pairs["Drug1"].map(drug_to_row)
    pairs["drug2_id"] = pairs["Drug2"].map(drug_to_row)
    assert pairs[["drug1_id", "drug2_id"]].notna().all().all()
    pairs.to_csv(OUT / f"{name}_pairs.csv", index=False)

# ---------- Reproduce using SAVED raw values and parameters ----------
with np.load(OUT / "raw_molecular_features.npz", allow_pickle=False) as saved:
    check_raw = saved["rdkit_raw"]
    check_fp = saved["morgan"]
    np.testing.assert_array_equal(saved["all_drugs"], np.asarray(all_drugs))

with np.load(OUT / "preprocessing_parameters.npz", allow_pickle=False) as p:
    check_raw = check_raw.copy()
    saved_ipc = int(p["ipc_col"].item())
    finite = np.isfinite(check_raw[:, saved_ipc])
    check_raw[finite, saved_ipc] = np.log1p(check_raw[finite, saved_ipc])
    check_raw[~np.isfinite(check_raw)] = np.nan

    check_available = check_raw[:, p["available_indices"]]
    check_imputed = np.where(
        np.isfinite(check_available), check_available, p["medians"]
    )
    check_selected = check_imputed[:, p["variable_mask"]]
    check_scaled = (
        check_selected - p["scaler_mean"]
    ) / p["scaler_scale"]

    reproduced = np.concatenate(
        [check_fp.astype(np.float64), check_scaled], axis=1
    ).astype(np.float32)

with np.load(OUT / "molecular_features.npz", allow_pickle=False) as saved:
    np.testing.assert_array_equal(reproduced, saved["x"])

metadata = {
    "protocol_folder": str(PROTOCOL),
    "rdkit": rdBase.rdkitVersion,
    "numpy": np.__version__,
    "sklearn": sklearn.__version__,
    "morgan": {
        "radius": 2, "bits": 2048, "includeChirality": False,
        "implementation": "rdFingerprintGenerator.GetMorganGenerator",
    },
    "registry_sources": [
        {"path": str(p), "sha256": sha256_file(p)}
        for p in registry_sources
    ],
    "raw_descriptor_count": 208,
    "retained_descriptor_count": len(retained_names),
    "all_nonfinite_train_dropped": all_missing_dropped,
    "zero_variance_train_dropped": zero_variance_dropped,
    "fixed_transformation": "log1p on finite nonnegative Ipc only",
    "nonfinite_policy": "Treat nonfinite descriptors as missing",
    "imputation": "Training-drug median",
    "variance_selection": "Training-drug variance > 0",
    "scaling": "StandardScaler fitted on training drugs only",
    "train_fit_drugs": len(train_rows),
    "cached_drugs": len(all_drugs),
    "feature_shape": list(x_new.shape),
    "maximum_absolute_value": float(np.abs(x_new).max()),
    "saved_reproduction_passed": True,
    "historical_feature_identity_verified": False,
    "training_performed": False,
    "files_sha256": {
        p.name: sha256_file(p)
        for p in sorted(OUT.iterdir()) if p.is_file()
    },
}
(OUT / "feature_metadata.json").write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

# Back up protocol + features together.
zip_path = Path(shutil.make_archive(
    str(PROTOCOL) + "_WITH_FEATURES", "zip", root_dir=PROTOCOL
))
with zipfile.ZipFile(zip_path) as archive:
    assert archive.testzip() is None, "ZIP integrity failure"

SCAFFOLD_FEATURE_DIR = OUT

print("\nTRAIN-ONLY FEATURES — COMPLETED")
print("Train fit drugs:", len(train_rows))
print("Cached drugs:", len(all_drugs))
print("RDKit descriptors retained:", len(retained_names))
print("Final feature shape:", x_new.shape)
print("All-nonfinite Train dropped:", all_missing_dropped)
print("Zero-variance Train dropped:", zero_variance_dropped)
print("Final NaN:", int(np.isnan(x_new).sum()))
print("Final Inf:", int(np.isinf(x_new).sum()))
print("Maximum absolute feature value:", float(np.abs(x_new).max()))
print(" Saved-feature reproduction passed")
print(" ZIP integrity passed")
print(" No training performed")
print("Feature folder:", OUT)

display(FileLink(str(zip_path.relative_to("/kaggle/working"))))